## Import libraries and healthiar

In [1]:
## Load OS module
import os

## Add environment variable for R installation
#os.environ['R_HOME'] = r"C:\Program Files\R\R-4.4.1"
os.environ["PATH"] = r"C:\Users\ArPa3547\AppData\Local\Programs\R\R-4.6.0\bin\x64"

## Import pry2 functions
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import FloatVector, IntVector,globalenv,ListVector, DataFrame, StrVector, BoolVector
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.rinterface import rternalize

## Import other modules
import pandas as pd
import geopandas as gpd
import matplotlib as plt
import xarray as xr
import rioxarray
from scipy import interpolate
from types import SimpleNamespace

## Import healthiar as Python wrapper
healthiar = importr("healthiar")

In [2]:
# Function to convert rpy2 lists and dataframes to python lists and pandas dataframes
def rpy2_to_py(obj):
    """
    Recursive conversion
    """
    if isinstance(obj, DataFrame):
        with localconverter(ro.default_converter + pandas2ri.converter):
            return ro.conversion.rpy2py(obj)
    
    elif isinstance(obj, ListVector):
        return {name: rpy2_to_py(obj.rx2(name)) for name in obj.names}
    
    elif isinstance(obj, (IntVector, FloatVector, StrVector, BoolVector)):
        if len(obj) == 1:
            return obj[0]  # single elements
        else:
            return list(obj)  # multiple elements
    else:
        return obj  # fallback for other types


## Perform tests
### List/tuple input

In [ ]:
# Get Python data
exp_central = [20, 20]
prop_pop_exp = [0.5, 0.5]
bhd_central = [10]

# Convert to R data
r_exp_central = FloatVector(exp_central)
r_prop_pop_exp = FloatVector(prop_pop_exp)
r_bhd_central = FloatVector(bhd_central)

# Call healthiar function
result = healthiar.attribute_health(
    exp_central = r_exp_central,
    prop_pop_exp = r_prop_pop_exp,
    cutoff_central = 5,
    rr_central = 1.08,
    rr_increment = 10,
    erf_shape = "linear_log",
    bhd_central = r_bhd_central
)

## Verify result
expected = 0.927071
print(round(result.rx2('health_main').rx2('impact')[0], 6))
print(round(result.rx2('health_main').rx2('impact')[0], 6) == expected)

0.927071
True
<class 'rpy2.robjects.vectors.ListVector'>
<class 'pandas.DataFrame'>


In [6]:
# Get Python data
exp_central = (20, 20)
prop_pop_exp = (0.5, 0.5)

# Convert to R data
r_exp_central = FloatVector(exp_central)
r_prop_pop_exp = FloatVector(prop_pop_exp)

# Call healthiar function
result = healthiar.attribute_health(
    exp_central = r_exp_central,
    prop_pop_exp = r_prop_pop_exp,
    cutoff_central = 5,
    rr_central = 1.08,
    rr_increment = 10,
    erf_shape = "log_log",
    bhd_central = 10
)

## Verify result
expected = 0.936215963
print(round(result.rx2('health_main').rx2('impact')[0], 9))
print(round(result.rx2('health_main').rx2('impact')[0], 9) == expected)

0.936215963
True


### pandas DataFrame input

In [6]:
# Get Python data
data = pd.DataFrame({
    "mean_concentration": [8.85],
    "cut_off_value": [5],
    "incidents_per_100_000_per_year": [357.27],
    "population_at_risk": [8606096],
    "relative_risk": [1.369],
    "pollutant": ["PM2.5"],
    "evaluation_name": ["GeLuft_COPD"],
    "estimated_number_of_attributable_cases_central": [3502]
})

# Convert to R data
r_exp_central = FloatVector(data["mean_concentration"])
r_cutoff_central = FloatVector(data["cut_off_value"])
r_bhd_central = FloatVector(data["incidents_per_100_000_per_year"] / 10**5 * data["population_at_risk"])
r_rr_central = FloatVector(data["relative_risk"])


# Call healthiar function
result = healthiar.attribute_health(
    approach_risk = "relative_risk",
    exp_central = r_exp_central,
    cutoff_central = r_cutoff_central,
    bhd_central = r_bhd_central,
    rr_central = r_rr_central,
    rr_increment = 10,
    erf_shape = "log_linear"
)

## Verify result
expected = data["estimated_number_of_attributable_cases_central"]
print(round(result.rx2('health_main').rx2('impact')[0], 0))
print(round(result.rx2('health_main').rx2('impact')[0], 0) == expected)

3502.0
0    True
Name: estimated_number_of_attributable_cases_central, dtype: bool


### Function input

In [ ]:
## Get Python data
data = pd.read_csv("data/LMU_O3_COPD_mort_2015_2016.csv", skiprows = [1])

## Convert to R data
from rpy2.rinterface import initr

initr()

from rpy2.rinterface import rternalize

@rternalize
def CubicSpline(x, y):
    return interpolate.CubicSpline(x, y)
 
r_prop_exp = FloatVector(data["Population.affected"])
r_exp_central = FloatVector(data["Mean.O3"])
r_bhd_central =  FloatVector(data["bhd"])
r_geo_id_micro = FloatVector(data["X"])

## Call healthiar function
result = healthiar.attribute_health(
        erf_eq_central = CubicSpline(data["x"][0:20], data["y"][0:20]),
        erf_eq_lower = CubicSpline(data["x"][0:20], data["y_l"][0:20]),
        erf_eq_upper = CubicSpline(data["x"][0:20], data["y_u"][0:20]),
        prop_pop_exp = r_prop_exp,
        exp_central = r_exp_central, # exposure distribution for ozone
        cutoff_central = 0,
        bhd_central =  r_bhd_central, #COPD mortality in Germany 2015 and 2016
        geo_id_micro = r_geo_id_micro
)

## Verify result
## Errors, need to further investigate rternalize function in low-level interface

ValueError: Not an rpy2 R object and unable to map it to one: 0      66.05
1      68.05
2      70.05
3      72.05
4      74.05
5      76.05
6      78.05
7      80.05
8      82.05
9      84.05
10     86.05
11     88.05
12     90.05
13     92.05
14     94.05
15     96.05
16     98.05
17    100.05
18    102.05
19    104.05
Name: x, dtype: float64

### Spatial format input

In [11]:
## Get python data
poll_grid = xr.open_dataset("data/pm25.tif", engine = "rasterio", masked = True)
#poll_grid["band_data"].plot()
print(type(poll_grid))

geo_units = gpd.read_file("data/municipalities_brussels.gpkg")
#geo_units.head()
#geo_units.plot(facecolor = "none")
print(type(geo_units))

<class 'xarray.core.dataset.Dataset'>
<class 'geopandas.geodataframe.GeoDataFrame'>


### Output reuse

In [ ]:
## Get Python data

## Convert to R data
  
## Call healthiar function
output_attribute_scen_1 = healthiar.attribute_health(
  exp_central = 8.85,
  cutoff_central = 5,
  bhd_central = 25000,
  approach_risk = "relative_risk",
  erf_shape = "log_linear",
  rr_central = 1.118, rr_lower = 1.060, rr_upper = 1.179,
  rr_increment = 10,
  info = "PM2.5_mortality_2010"
)

output_attribute_scen_2 = healthiar.attribute_health(
  exp_central = 6,
  cutoff_central = 5,
  bhd_central = 25000,
  approach_risk = "relative_risk",
  erf_shape = "log_linear",
  rr_central = 1.118, rr_lower = 1.060, rr_upper = 1.179,
  rr_increment = 10,
  info = "PM2.5_mortality_2020"
)

result = healthiar.compare(
  output_attribute_scen_1 = output_attribute_scen_1,
  output_attribute_scen_2 = output_attribute_scen_2,
  approach_comparison = "delta"
)

## Verify result
expected = [774, 409, 1127]
print(list(map(round, list(result.rx2('health_main').rx2('impact')))))
print(list(map(round, list(result.rx2('health_main').rx2('impact')))) == expected)

[774, 409, 1127]
True


### Access healthiar documentation

In [8]:
?healthiar.attribute_health

Signature:       healthiar.attribute_health(*args, **kwargs)
Type:            DocumentedSTFunction
String form:    
function(
           # RR & AR
           approach_risk = "relative_risk",
           exp_central, exp_lower = NULL, e <...> rgs)
           
           return(output)
           
           
           }
           <bytecode: 0x00000230ee920140>
           <environment: namespace:healthiar>
           
File:            c:\users\arpa3547\appdata\local\programs\python\python314\lib\site-packages\rpy2\robjects\functions.py
Docstring:      
Wrapper around an R function.

The docstring below is built from the R documentation.

description
-----------


 This function calculates the attributable health impacts (mortality or morbidity) due to
 exposure to an environmental stressor (air pollution or noise), using either relative risk ( RR ) or absolute risk ( AR ).
 
 Arguments for both  RR & AR  pathways
 
     approach_risk 
     exp_central ,  exp_lower ,  exp_upper 
     cut